<a href="https://colab.research.google.com/github/worldterminator/worldterminator/blob/main/diss%20data%20ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

311 OPCD -Present

In [1]:
import requests
import pandas as pd

In [2]:
API_URL = "https://data.nola.gov/resource/2jgv-pqrq.json"

KEEP_COLS = [
    "service_request",
    "request_type",
    "date_created",
    "case_close_date",
    "request_status",
    "responsible_agency",
    "address_councildis",
    "rowid",
    "final_x",
    "final_y",
    "longitude",
    "latitude",
    "geocoded_column",
]

test endpoint and the wanted fields

In [3]:
params = {

    "$select": ",".join(KEEP_COLS),

    "$limit": 10

}
r = requests.get(API_URL, params=params, timeout=120)

r.raise_for_status()

df_test = pd.DataFrame(r.json())
print(df_test.shape)
df_test.head()

(10, 13)


,service_request,request_type,date_created,case_close_date,request_status,responsible_agency,rowid,longitude,latitude,geocoded_column,address_councildis,final_x,final_y
0,2021-847416,Mayor's Request,2021-12-10T21:22:27.000,2022-07-08T09:30:48.000,Closed,Executive Office of the Mayor,847416,0.0,0.0,"{'latitude': '0.0', 'longitude': '0.0'}",NaN,NaN,NaN
1,2022-858955,Tax and Revenue,2022-02-04T15:39:45.000,2022-02-22T01:09:55.000,Closed,Bureau of Revenue,858955,0.0,0.0,"{'latitude': '0.0', 'longitude': '0.0'}",NaN,NaN,NaN
2,2024-1145120,Traffic Safety,2024-10-30T12:18:29.000,NaN,Pending,Department of Public Works,1145120,-90.12458247242685,29.978768001649772,"{'latitude': '29.978768001649772', 'longitude'...",A,3663515.88941,539789.54625
3,2024-1145799,Trash/Recycling,2024-11-02T07:58:56.000,2024-11-02T04:09:40.000,Closed,Department of Sanitation,1145799,-90.091290391251,29.940758205316882,"{'latitude': '29.940758205316882', 'longitude'...",B,3674205.35803,526080.457854
4,2024-1145838,Roads and Streets,2024-11-02T15:10:10.000,NaN,Pending,Department of Public Works,1145838,-90.10896265604138,29.988242136809447,"{'latitude': '29.988242136809447', 'longitude'...",A,3668424.02055,543287.21947


In [4]:
df_test.columns.tolist()

['service_request',
 'request_type',
 'date_created',
 'case_close_date',
 'request_status',
 'responsible_agency',
 'rowid',
 'longitude',
 'latitude',
 'geocoded_column',
 'address_councildis',
 'final_x',
 'final_y']

then full paginated request, for repeating

In [5]:
LIMIT = 50000
offset = 0
parts = []

while True:
    params = {
        "$select": ",".join(KEEP_COLS),
        "$where": "date_created >= '2012-01-01T00:00:00'", #from 2012 onward
        "$limit": LIMIT,
        "$offset": offset,
        "$order": "rowid"
    }

    r = requests.get(API_URL, params=params, timeout=120)
    r.raise_for_status()

    rows = r.json()

    if not rows:
        break

    batch = pd.DataFrame(rows)
    parts.append(batch)

    offset += len(batch)
    print(f"Downloaded {offset:,} rows")

df_311 = pd.concat(parts, ignore_index=True)

print(f"\nDownload complete!")
print(f"Rows: {len(df_311):,}")
print(f"Columns: {len(df_311.columns)}")

Downloaded 50,000 rows
Downloaded 100,000 rows
Downloaded 150,000 rows
Downloaded 200,000 rows
Downloaded 250,000 rows
Downloaded 300,000 rows
Downloaded 350,000 rows
Downloaded 400,000 rows
Downloaded 450,000 rows
Downloaded 500,000 rows
Downloaded 550,000 rows
Downloaded 600,000 rows
Downloaded 650,000 rows
Downloaded 700,000 rows
Downloaded 750,000 rows
Downloaded 800,000 rows
Downloaded 850,000 rows
Downloaded 900,000 rows
Downloaded 950,000 rows
Downloaded 1,000,000 rows
Downloaded 1,020,471 rows

Download complete!
Rows: 1,020,471
Columns: 13


In [6]:
print("Shape:", df_311.shape)

print("\nDate range:")
print(df_311["date_created"].min(), "to", df_311["date_created"].max())

print("\nDuplicate IDs:")
print("rowid:", df_311["rowid"].duplicated().sum())
print("service_request:", df_311["service_request"].duplicated().sum())

print("\nMissingness:")
print(df_311.isna().sum().sort_values(ascending=False))

print("\nTop request types:")
print(df_311["request_type"].value_counts(dropna=False).head(20))

lon = pd.to_numeric(df_311["longitude"], errors="coerce")
lat = pd.to_numeric(df_311["latitude"], errors="coerce")

valid_coord = (
    lon.notna() &
    lat.notna() &
    (lon != 0) &
    (lat != 0)
)

print("\nCoordinate check:")
print("Valid coordinates:", valid_coord.sum())
print("Valid coordinate %:", f"{valid_coord.mean():.2%}")
print("0,0 coordinates:", ((lon == 0) & (lat == 0)).sum())

Shape: (1020471, 13)

Date range:
2019-01-01T21:33:04.000 to 2026-08-17T23:24:34.000

Duplicate IDs:
rowid: 0
service_request: 0

Missingness:
address_councildis    329015
case_close_date       232542
final_y                40891
final_x                40891
responsible_agency      6635
request_type            1757
request_status             0
service_request            0
date_created               0
rowid                      0
longitude                  0
latitude                   0
geocoded_column            0
dtype: int64

Top request types:
request_type
Trash/Recycling                                475160
Property Maintenance                            99096
Abandoned Vehicles                              91900
Roads/Drainage                                  63309
Traffic Signals/Signs/Striping/Streetlights     58269
Streetlights                                    36963
Parks & Parkways                                31019
Roads and Streets                               29541
Dr

In [7]:
df_311.to_parquet("311_OPCD_2019plus_raw_selected.parquet", index=False)

In [8]:
from google.colab import files
files.download("311_OPCD_2019plus_raw_selected.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
311 OPCD 2012-2018

In [9]:
API_URL_ARCHIVE = "https://data.nola.gov/resource/3iz8-nghx.json"

KEEP_COLS_ARCHIVE = [
    "ticket_id",
    "issue_type",
    "ticket_created_date_time",
    "ticket_closed_date_time",
    "ticket_status",
    "council_district",
    "longitude",
    "latitude",
    "location",
    "geom",
]

In [10]:
params = {
    "$select": ",".join(KEEP_COLS_ARCHIVE),
    "$limit": 10
}

r = requests.get(API_URL_ARCHIVE, params=params, timeout=120)
r.raise_for_status()

df_311_archive_test = pd.DataFrame(r.json())

print(df_311_archive_test.shape)
df_311_archive_test.head()

(10, 9)


,ticket_id,issue_type,ticket_created_date_time,ticket_closed_date_time,ticket_status,council_district,longitude,latitude,location
0,101000847991,Street Light,2018-08-21T14:03:55.000,2018-08-28T09:38:35.000,Closed,A,-90.1157700850457,29.9202224971517,"{'latitude': '29.9202224971517', 'longitude': ..."
1,101000847903,Large Item Trash/Garbage Pickup,2018-08-21T12:29:40.000,2018-08-28T10:51:27.000,Closed,B,-90.1019344489206,29.950804540038,"{'latitude': '29.950804540038', 'longitude': '..."
2,101000849742,Pothole/Roadway Surface Repair,2018-08-24T14:41:00.000,2018-08-28T09:38:50.000,Closed,E,-90.0028355253192,30.0271568286374,"{'latitude': '30.0271568286374', 'longitude': ..."
3,101000847653,Trash/Garbage Pickup,2018-08-20T16:28:36.000,2018-08-28T14:03:28.000,Closed,A,-90.1045471973296,29.9699067477013,"{'latitude': '29.9699067477013', 'longitude': ..."
4,101000850848,Code Enforcement General Request,2018-08-28T11:34:00.000,2018-08-28T13:02:24.000,Closed,C,-90.0491191933234,29.9446163523183,"{'latitude': '29.9446163523183', 'longitude': ..."


In [11]:
LIMIT = 50000
offset = 0
parts = []

while True:
    params = {
        "$select": ",".join(KEEP_COLS_ARCHIVE),
        "$limit": LIMIT,
        "$offset": offset,
        "$order": "ticket_created_date_time,ticket_id"
    }

    r = requests.get(API_URL_ARCHIVE, params=params, timeout=120)
    r.raise_for_status()

    rows = r.json()

    if not rows:
        break

    batch = pd.DataFrame(rows)
    parts.append(batch)

    offset += len(batch)
    print(f"Downloaded {offset:,} rows")

df_311_archive = pd.concat(parts, ignore_index=True)

print("\nDownload complete!")
print(f"Rows: {len(df_311_archive):,}")
print(f"Columns: {len(df_311_archive.columns)}")

Downloaded 50,000 rows
Downloaded 100,000 rows
Downloaded 150,000 rows
Downloaded 200,000 rows
Downloaded 250,000 rows
Downloaded 300,000 rows
Downloaded 303,735 rows

Download complete!
Rows: 303,735
Columns: 9


In [12]:
print("Shape:", df_311_archive.shape)

print("\nDate range:")
print(
    df_311_archive["ticket_created_date_time"].min(),
    "to",
    df_311_archive["ticket_created_date_time"].max()
)

print("\nDuplicate ticket_id:")
print(df_311_archive["ticket_id"].duplicated().sum())

print("\nMissingness:")
print(df_311_archive.isna().sum().sort_values(ascending=False))

print("\nTop issue types:")
print(
    df_311_archive["issue_type"]
    .value_counts(dropna=False)
    .head(20)
)

lon = pd.to_numeric(df_311_archive["longitude"], errors="coerce")
lat = pd.to_numeric(df_311_archive["latitude"], errors="coerce")

valid_coord = (
    lon.notna() &
    lat.notna() &
    (lon != 0) &
    (lat != 0)
)

print("\nCoordinate check:")
print("Valid coordinates:", valid_coord.sum())
print("Valid coordinate %:", f"{valid_coord.mean():.2%}")
print("0,0 coordinates:", ((lon == 0) & (lat == 0)).sum())

Shape: (303735, 9)

Date range:
2012-03-12T09:46:20.000 to 2019-01-02T14:28:00.000

Duplicate ticket_id:
1321

Missingness:
ticket_closed_date_time     25956
council_district            12807
ticket_id                       0
ticket_created_date_time        0
issue_type                      0
ticket_status                   0
longitude                       0
latitude                        0
location                        0
dtype: int64

Top issue types:
issue_type
Code Enforcement General Request       45126
Street Light                           36732
Trash/Garbage Pickup                   35680
Abandoned Vehicle Reporting/Removal    31798
Residential Recycling Programs         27501
Large Item Trash/Garbage Pickup        25070
Pothole/Roadway Surface Repair         16924
General Service Request                13195
Illegal Dumping Reporting              11764
Catch Basin Maintenance                 9480
Street Flooding/Drainage                9169
Tree Service                     

In [13]:
dup = df_311_archive[ #given ticked_id duplicates
    df_311_archive["ticket_id"].duplicated(keep=False)
].sort_values("ticket_id")

dup.head(20)

,ticket_id,issue_type,ticket_created_date_time,ticket_closed_date_time,ticket_status,council_district,longitude,latitude,location
78,101000001000,Pothole/Roadway Surface Repair,2012-03-19T10:05:50.000,2012-11-08T16:24:34.000,Closed,B,-90.102347586828,29.9180961547502,"{'latitude': '29.9180961547502', 'longitude': ..."
79,101000001000,Pothole/Roadway Surface Repair,2012-03-19T10:05:50.000,2012-11-08T16:24:34.000,Closed,B,-90.102347586828,29.9180961547502,"{'latitude': '29.9180961547502', 'longitude': ..."
153,101000001441,Street Light,2012-03-21T09:52:21.000,2012-11-08T17:04:05.000,Closed,D,-90.0377474972926,29.9707357610986,"{'latitude': '29.9707357610986', 'longitude': ..."
154,101000001441,Street Light,2012-03-21T09:52:21.000,2012-11-08T17:04:05.000,Closed,D,-90.0377474972926,29.9707357610986,"{'latitude': '29.9707357610986', 'longitude': ..."
367,101000002297,Street Flooding/Drainage,2012-03-23T16:41:37.000,2012-04-24T08:59:21.000,Closed,A,-90.1041193534018,29.9794484455347,"{'latitude': '29.9794484455347', 'longitude': ..."
368,101000002297,Street Flooding/Drainage,2012-03-23T16:41:37.000,2012-04-24T08:59:21.000,Closed,A,-90.1041193534018,29.9794484455347,"{'latitude': '29.9794484455347', 'longitude': ..."
804,101000003641,Street Light,2012-03-30T12:09:09.000,2012-11-08T17:04:17.000,Closed,D,-90.0375946773703,29.9704552414518,"{'latitude': '29.9704552414518', 'longitude': ..."
805,101000003641,Street Light,2012-03-30T12:09:09.000,2012-11-08T17:04:17.000,Closed,D,-90.0375946773703,29.9704552414518,"{'latitude': '29.9704552414518', 'longitude': ..."
1224,101000004828,Street Light,2012-04-05T09:30:21.000,2013-10-16T17:01:25.000,Closed,A,-90.1320451686034,29.9433499480254,"{'latitude': '29.9433499480254', 'longitude': ..."
1223,101000004828,Street Light,2012-04-05T09:30:21.000,2013-10-16T17:01:25.000,Closed,A,-90.1320451686034,29.9433499480254,"{'latitude': '29.9433499480254', 'longitude': ..."


In [16]:
cols = [
    "ticket_id",
    "issue_type",
    "ticket_created_date_time",
    "ticket_closed_date_time",
    "ticket_status",
    "council_district",
    "longitude",
    "latitude"
]

dup = df_311_archive[df_311_archive["ticket_id"].duplicated(keep=False)]
dup.duplicated(subset=cols).sum()

np.int64(1230)

ok,so 91 are not exact duplicates—at least differ from the other record with same id in one checked field